### LSTM baseline

La prima versione del modello utilizza una LSTM singola seguita da uno strato fully connected.
Questa architettura rappresenta una baseline standard per problemi di regressione su serie temporali.

Il modello è addestrato utilizzando la Mean Squared Error (MSE) come funzione di loss, che penalizza fortemente errori elevati.
Questa scelta è appropriata in presenza di dati stazionari e con distribuzioni simili tra training, validation e test set.

Tuttavia, in presenza di distribution shift e outlier nel test set, l’MSE tende a produrre valori di loss molto elevati,
rendendo difficile la generalizzazione del modello.


In [ ]:
from tensorflow.keras import layers
import tensorflow as tf

def build_lstm_baseline(input_width, num_features, num_targets,
                        units=64, dropout=0.2, dense_units=64, lr=1e-3):

    model = tf.keras.Sequential([
        layers.Input(shape=(input_width, num_features)),
        layers.LSTM(units, dropout=dropout, recurrent_dropout=0.0),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(num_targets),
        layers.Reshape((1, num_targets))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    return model


### LSTM migliorata

L’analisi statistica dei dati ha evidenziato una significativa differenza di distribuzione tra validation e test set,
con una maggiore varianza e presenza di outlier nel test.

Per affrontare questo scenario, il modello è stato modificato secondo tre principi:

1. Aumento moderato della capacità tramite una LSTM stacked, per migliorare l’estrazione delle rappresentazioni temporali.
2. Introduzione di Layer Normalization e gradient clipping per stabilizzare l’ottimizzazione.
3. Sostituzione della MSE con la Huber loss, più robusta agli outlier e alle variazioni improvvise.

Queste modifiche non mirano ad aumentare indiscriminatamente la complessità del modello,
ma a renderlo più robusto in presenza di dinamiche non stazionarie.


In [ ]:
from tensorflow.keras import layers
import tensorflow as tf

def build_lstm_robust(input_width, num_features, num_targets,
                      units=128, dropout=0.2, dense_units=128, lr=5e-4):

    model = tf.keras.Sequential([
        layers.Input(shape=(input_width, num_features)),

        layers.LSTM(
            units,
            return_sequences=True,
            dropout=dropout,
            recurrent_dropout=0.0
        ),
        layers.Dropout(dropout),

        layers.LSTM(
            units,
            return_sequences=True,
            dropout=dropout,
            recurrent_dropout=0.0
        ),
        layers.Dropout(dropout),

        layers.LSTM(
            units // 2,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0
        ),

        layers.LayerNormalization(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dropout(dropout),

        layers.Dense(num_targets),
        layers.Reshape((1, num_targets))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=["mae"]
    )
    return model


### Motivazioni delle modifiche all’architettura LSTM

L’analisi dei risultati ottenuti con la LSTM baseline ha evidenziato alcune criticità riconducibili non tanto a un errore di implementazione, quanto alle caratteristiche intrinseche del dataset. In particolare, il test set presenta una maggiore variabilità e una distribuzione differente rispetto al training e alla validation. A partire da queste osservazioni sono state introdotte tre modifiche principali.

#### 1. Aumento moderato della capacità tramite LSTM stacked
La versione iniziale del modello utilizzava una singola LSTM, che tendeva a produrre predizioni eccessivamente semplificate. Questo comportamento è indicativo di una capacità limitata nell’estrarre rappresentazioni temporali articolate. L’introduzione di una LSTM stacked consente di separare l’estrazione di pattern temporali locali dai livelli di astrazione più alti, migliorando la qualità delle rappresentazioni senza incrementare eccessivamente la complessità del modello. L’obiettivo non è stato aumentare la memoria di lungo periodo, ma affinare la modellazione delle dinamiche presenti nella finestra temporale osservata. Due LSTM non servono ad aumentare la memoria, ma a creare una gerarchia di rappresentazioni temporali, migliorando l’estrazione e la combinazione dei pattern all’interno della finestra osservata.

#### 2. Introduzione di Layer Normalization e gradient clipping
L’elevata variabilità del test set e la presenza di outlier hanno reso evidente una certa instabilità durante l’ottimizzazione. Il gradient clipping è stato introdotto per limitare l’effetto di gradienti eccessivamente grandi, evitando aggiornamenti troppo aggressivi dei pesi in presenza di errori elevati. La Layer Normalization è stata aggiunta per stabilizzare le attivazioni interne del modello, rendendo l’addestramento meno sensibile a cambiamenti di scala e a fluttuazioni nella distribuzione dei dati. Insieme, queste tecniche contribuiscono a rendere il processo di apprendimento più stabile e controllato.
Durante le prime fasi di sperimentazione è stato osservato che il processo di addestramento risultava instabile. In particolare, la loss tendeva a diminuire inizialmente per poi risalire, i risultati erano sensibili alla scelta del seed e le prestazioni variavano in modo significativo tra diverse esecuzioni. Questi segnali indicano che il problema non era legato esclusivamente alla capacità del modello, ma al comportamento dell’ottimizzatore in presenza di esempi difficili.

In presenza di outlier o di variazioni improvvise nella distribuzione dei dati, i gradienti possono assumere valori molto elevati, producendo aggiornamenti dei pesi eccessivamente aggressivi. Il gradient clipping è stato quindi introdotto per limitare l’ampiezza degli aggiornamenti, consentendo al modello di correggere gli errori senza reagire in modo sproporzionato. L’obiettivo non è ridurre la capacità di apprendimento, ma rendere il processo di ottimizzazione più controllato e stabile.

Parallelamente, l’analisi delle distribuzioni ha evidenziato differenze di scala tra training, validation e test set. Questo comporta variazioni significative nelle attivazioni interne della rete, che possono cambiare regime quando la distribuzione dei dati si sposta. La Layer Normalization è stata introdotta per stabilizzare tali attivazioni, normalizzandole all’interno di ciascun esempio e riducendo la sensibilità del modello a fluttuazioni di scala.

Insieme, gradient clipping e Layer Normalization agiscono su due aspetti complementari dell’addestramento: il primo controlla l’intensità degli aggiornamenti dei pesi, mentre la seconda stabilizza la rappresentazione interna dei dati. Questa combinazione ha permesso di ottenere un processo di apprendimento più robusto in presenza di dinamiche non stazionarie e outlier.
#### 3. Sostituzione della MSE con la Huber loss
La Mean Squared Error penalizza in modo quadratico gli errori elevati, rendendola particolarmente sensibile agli outlier. Poiché il test set presenta una varianza significativamente maggiore rispetto alla validation, l’uso della MSE portava a valori di loss molto elevati e poco rappresentativi della qualità media delle predizioni. La Huber loss combina il comportamento della MSE per errori piccoli con quello della MAE per errori grandi, riducendo l’influenza degli outlier sull’ottimizzazione. Questa scelta ha permesso di migliorare la robustezza del modello e di ottenere una valutazione più coerente delle prestazioni sul test set.

Nel complesso, le modifiche introdotte non mirano ad aumentare arbitrariamente la complessità della rete, ma a renderla più adatta alle caratteristiche del dataset, migliorando la stabilità dell’addestramento e la capacità di generalizzazione in presenza di dinamiche non stazionarie.
